# LFP explorer — traces, spectrograms, and coupling over time

Stage 3 collapsed each recording to one row. This notebook looks at the **same clean `.npz` files as a time series**: stacked LFP, spectrograms, and 10 s coherence / wPLI.

It does not need `windows.parquet`. Windows for the selected recording are computed in memory.

Run with **Restart & Run All**. Change `STEM` in the config cell to inspect another file.


## Setup

Paths match `data_ingestion.ipynb`. Constants come from `python/lfp_config.py` so band edges and window length stay identical to Stage 6.


In [ ]:
%matplotlib inline

from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal

PROJECT = Path("/Users/prestondavis/Desktop/Capstone")
if not (PROJECT / "derived" / "manifest.csv").exists():
    PROJECT = Path.cwd() if (Path.cwd() / "derived").is_dir() else Path.cwd().parent

sys.path.insert(0, str(PROJECT / "python"))

from IPython.display import display
from lfp_config import BANDS, FMAX_SPEC, FMIN_SPEC, REGIONS, WIN_S
from lfp_io import load_clean
from lfp_windows import window_features

DERIVED = PROJECT / "derived"
CLEAN = DERIVED / "clean"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 9,
})


In [ ]:
# --- pick one recording; everything below follows this stem ---
STEM = "Psi_C1_psi_F1_EZM"
ZOOM_START_S = 20.0  # start of the 10 s waveform zoom

MAN = pd.read_csv(DERIVED / "manifest.csv")
QC = pd.read_csv(DERIVED / "qc.csv")
EXCL = pd.read_csv(DERIVED / "exclusions.csv") if (DERIVED / "exclusions.csv").exists() else pd.DataFrame()

assert (CLEAN / f"{STEM}.npz").is_file(), f"missing {CLEAN / f'{STEM}.npz'}"
row = MAN.loc[MAN.stem == STEM]
assert len(row) == 1, STEM
ROW = row.iloc[0]
print(ROW[["stem", "cohort", "group", "drug", "mouse", "rectype", "subject_uid"]].to_string())


## Channel coverage

Each recording is a 3-node montage (BLA, vHPC, mPFC). Dead and high-kurtosis channels cannot enter a pair analysis. Color: live (0), high kurtosis (1), dead (2), missing (3).


In [ ]:
def coverage_matrix(qc: pd.DataFrame, excl: pd.DataFrame) -> pd.DataFrame:
    """Integer status per (stem, region) for a heatmap."""
    status = pd.DataFrame({"stem": qc["stem"]})
    for r in REGIONS:
        col = f"{r}_present"
        present = qc[col].fillna(False).astype(bool) if col in qc.columns else True
        status[r] = np.where(present, 0, 3)
    if excl is not None and not excl.empty:
        rank = {"high_kurtosis": 1, "dead_channel": 2, "missing": 3}
        for rec in excl.itertuples(index=False):
            if rec.region not in REGIONS:
                continue
            hit = status.stem == rec.stem
            status.loc[hit, rec.region] = np.maximum(
                status.loc[hit, rec.region], rank.get(rec.reason, 1)
            )
    return status.merge(qc[["stem", "cohort", "group", "rectype"]], on="stem")


cov = coverage_matrix(QC, EXCL)
print(cov[REGIONS].apply(pd.Series.value_counts).fillna(0).astype(int).to_string())
print(f"\nSelected stem statuses: {cov.loc[cov.stem == STEM, list(REGIONS)].iloc[0].to_dict()}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=False)
for ax, (cohort, sub) in zip(axes.ravel(), cov.groupby("cohort", sort=True)):
    mat = sub.set_index("stem")[list(REGIONS)].to_numpy()
    im = ax.imshow(mat, aspect="auto", vmin=0, vmax=3, cmap="viridis")
    ax.set_xticks(range(len(REGIONS)))
    ax.set_xticklabels(REGIONS)
    ax.set_yticks([])
    ax.set_title(f"{cohort}  n={len(sub)} files")
    ax.set_ylabel("recordings")
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, ticks=[0, 1, 2, 3],
             label="0 live / 1 high kurtosis / 2 dead / 3 missing")
fig.suptitle("Channel coverage by cohort", fontweight="bold")
fig.tight_layout()


## Load one recording

Stage-2 output: `float` LFP at 500 Hz and a boolean artifact mask per region (True = bad).


In [ ]:
lfp, msk, fs = load_clean(CLEAN / f"{STEM}.npz")
n = next(iter(lfp.values())).shape[-1]
t = np.arange(n) / fs
print(f"fs={fs:.1f} Hz  N={n}  duration={n / fs:.1f} s  regions={list(lfp)}")
for r in REGIONS:
    if r not in lfp:
        print(f"  {r}: missing")
        continue
    bad = float(msk[r].mean()) if r in msk else float("nan")
    print(f"  {r}: peak={np.max(np.abs(lfp[r])):.1f}  artifact={100 * bad:.2f}%")


## Stacked traces

Full session (downsampled for drawing) and a 10 s zoom. Shaded spans are artifact masks. Identical-looking traces across regions are a volume-conduction warning.


In [ ]:
def shade_mask(ax, t, mask, color="0.6"):
    if mask is None or not np.any(mask):
        return
    d = np.diff(mask.astype(int), prepend=0, append=0)
    starts = np.flatnonzero(d == 1)
    ends = np.flatnonzero(d == -1)
    for s, e in zip(starts, ends):
        ax.axvspan(t[min(s, len(t) - 1)], t[min(e - 1, len(t) - 1)], color=color, alpha=0.35, lw=0)


step = max(1, int(fs // 50))  # ~50 Hz drawing rate for the overview
fig, axes = plt.subplots(len(REGIONS), 2, figsize=(12, 7), sharex=False)
t0, t1 = ZOOM_START_S, ZOOM_START_S + WIN_S
z = (t >= t0) & (t < t1)
for i, r in enumerate(REGIONS):
    if r not in lfp:
        axes[i, 0].set_visible(False)
        axes[i, 1].set_visible(False)
        continue
    x = lfp[r]
    mask = msk.get(r)
    axes[i, 0].plot(t[::step], x[::step], color="0.15", lw=0.4)
    shade_mask(axes[i, 0], t, mask)
    axes[i, 0].set_ylabel(f"{r}\nµV")
    axes[i, 0].axvspan(t0, t1, color="tab:orange", alpha=0.15, lw=0)
    axes[i, 1].plot(t[z], x[z], color="0.15", lw=0.6)
    shade_mask(axes[i, 1], t, mask)
    axes[i, 1].set_xlim(t0, t1)
axes[0, 0].set_title("Full session (downsampled)")
axes[0, 1].set_title(f"Zoom {t0:.0f}–{t1:.0f} s")
axes[-1, 0].set_xlabel("time (s)")
axes[-1, 1].set_xlabel("time (s)")
fig.suptitle(f"{STEM}  stacked LFP", fontweight="bold")
fig.tight_layout()


## Spectrograms

Power in dB, 1–100 Hz. Look for a vHPC theta peak, BLA beta structure, and whether mPFC is neural or line-noise / flat.


In [ ]:
nper = int(round(WIN_S * fs))
fig, axes = plt.subplots(1, len(REGIONS), figsize=(12, 3.4), sharey=True)
for ax, r in zip(axes, REGIONS):
    if r not in lfp:
        ax.set_visible(False)
        continue
    f, ts, Sxx = signal.spectrogram(lfp[r], fs=fs, nperseg=nper, noverlap=nper // 2)
    band = (f >= FMIN_SPEC) & (f <= FMAX_SPEC)
    db = 10 * np.log10(np.maximum(Sxx[band], 1e-20))
    im = ax.pcolormesh(ts, f[band], db, shading="auto", cmap="magma")
    ax.set_title(r)
    ax.set_xlabel("time (s)")
    ax.set_ylim(FMIN_SPEC, FMAX_SPEC)
axes[0].set_ylabel("frequency (Hz)")
fig.colorbar(im, ax=axes, fraction=0.02, label="power (dB)")
fig.suptitle(f"{STEM}  spectrograms ({WIN_S:.0f} s windows)", fontweight="bold")


## Coupling over time

Each point is one 10 s window (5 s hop). **wPLI** (solid) is the volume-conduction-robust number; coherence (dashed) is shown for comparison. Beta is highlighted; other bands are drawn lighter.

$$C_{xy}(f)=\frac{|G_{xy}(f)|^2}{G_{xx}(f)G_{yy}(f)}$$


In [ ]:
win = window_features(lfp, msk, fs)
print(f"kept windows: {len(win)}  of duration {n / fs:.1f} s")
display(win[["t_start", "t_end", "frac_artifact"]].describe().T)

pairs = ["BLA-vHPC", "BLA-mPFC", "vHPC-mPFC"]
tc = 0.5 * (win["t_start"] + win["t_end"])
fig, axes = plt.subplots(len(pairs), 1, figsize=(11, 7), sharex=True, sharey=True)
for ax, pair in zip(axes, pairs):
    for band in BANDS:
        wcol, ccol = f"{pair}_wpli_{band}", f"{pair}_coh_{band}"
        if wcol not in win:
            continue
        lw = 1.8 if band == "beta" else 0.9
        alpha = 1.0 if band == "beta" else 0.45
        ax.plot(tc, win[wcol], lw=lw, alpha=alpha, label=f"{band} wPLI")
        ax.plot(tc, win[ccol], lw=0.8, ls="--", alpha=alpha * 0.8, color="0.4")
    ax.set_ylabel(pair)
    ax.set_ylim(0, 1)
    ax.legend(ncol=5, fontsize=7, loc="upper right")
axes[-1].set_xlabel("time (s)")
fig.suptitle(f"{STEM}  10 s coupling (solid wPLI, dashed coherence)", fontweight="bold")
fig.tight_layout()


## Same mouse, both drug weeks (C1/C2)

If this stem belongs to a crossover cohort, plot BLA–vHPC beta wPLI for `BL1 | inj | EZM | BL2` on psilocybin vs saline. Other cohorts print a skip message.


In [ ]:
RECORDER = ["BL1", "inj", "EZM", "EPM", "BL2"]

if ROW.design != "within":
    print(f"Skip small multiples: {STEM} is {ROW.design} ({ROW.cohort}), not a C1/C2 crossover.")
else:
    sibs = MAN[(MAN.cohort == ROW.cohort) & (MAN.mouse == ROW.mouse)]
    drugs = [d for d in ("sal", "psi") if d in set(sibs.drug)]
    recs = [r for r in RECORDER if r in set(sibs.rectype)]
    fig, axes = plt.subplots(len(drugs), len(recs), figsize=(3.1 * len(recs), 2.6 * len(drugs)),
                             sharex=True, sharey=True, squeeze=False)
    for i, drug in enumerate(drugs):
        for j, rec in enumerate(recs):
            ax = axes[i, j]
            hit = sibs[(sibs.drug == drug) & (sibs.rectype == rec)]
            ax.set_title(f"{drug} {rec}", fontsize=9)
            if hit.empty:
                ax.set_visible(False)
                continue
            stem = hit.iloc[0].stem
            npz = CLEAN / f"{stem}.npz"
            if not npz.is_file():
                ax.text(0.5, 0.5, "no npz", ha="center", va="center", transform=ax.transAxes)
                continue
            L, M, fsi = load_clean(npz)
            w = window_features(L, M, fsi)
            col = "BLA-vHPC_wpli_beta"
            if w.empty or col not in w:
                continue
            tc = 0.5 * (w.t_start + w.t_end)
            ax.plot(tc, w[col], color="tab:blue" if drug == "psi" else "0.35", lw=1.2)
            ax.set_ylim(0, 1)
            if j == 0:
                ax.set_ylabel("BLA–vHPC β wPLI")
            if i == len(drugs) - 1:
                ax.set_xlabel("time (s)")
    fig.suptitle(f"{ROW.cohort} {ROW.mouse}  BLA–vHPC beta wPLI by session",
                 fontweight="bold")
    fig.tight_layout()


### Output of this step

- Coverage heatmap: live / high-kurtosis / dead / missing per file × region.
- One recording: stacked traces, spectrograms, 10 s wPLI/coherence for all three pairs.
- C1/C2 only: small multiples of BLA–vHPC β wPLI across the test day and both drug weeks.
- Batch window table and session mean/variance are produced by `step6_windows.py`, not this notebook.
